In [ ]:
!pip install transformers torch --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 98.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 58.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 56.7 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
import os
from pathlib import Path
import pandas as pd

# 1. List files in the root /content folder
print("Files in /content:")
print(os.listdir('/content'))

# 2. Search recursively for any CSV under /content
csv_files = list(Path('/content').rglob('*.csv'))
print("\nCSV files found:")
for path in csv_files:
    print(" ", path)

# 3. Load the dataset (adjust the path to whichever CSV you see)
dataset_path = '/content/cleaned_twitter_data.csv'  # or the full path you discovered
df_full = pd.read_csv(dataset_path)

# 4. Verify load
print("\nTotal tweets:", len(df_full))
print(df_full.columns)
print(df_full.head())


Files in /content:
['.config', 'cleaned_twitter_data.csv', 'sample_data']

CSV files found:
  /content/cleaned_twitter_data.csv
  /content/sample_data/mnist_test.csv
  /content/sample_data/california_housing_train.csv
  /content/sample_data/california_housing_test.csv
  /content/sample_data/mnist_train_small.csv

Total tweets: 279691
Index(['Twitter_User_Name', 'Twitter_Account', 'Twitter_User_Description',
       'Tweet_id', 'Tweet_created_at', 'Tweet_text', 'Label', 'Word_Count',
       'Url_Count', 'Retweet', 'Mentions_Count', 'Hashtags_Count',
       'QuesMark_Count', 'Exclamations_Count', 'SpecialCharacters_Count',
       'Nouns_Count', 'Pronouns_Count', 'Verb_Count', 'Adverb_Count',
       'Positive_Word_Ratio', 'Negative_Word_Ratio', 'Neutral_Word_Ratio',
       'Following', 'Followers', 'Verified', 'Link', 'Real_Location'],
      dtype='object')
  Twitter_User_Name Twitter_Account  \
0        Museum Bot       MuseumBot   
1        Museum Bot       MuseumBot   
2        Museum B

In [ ]:
import re

# Define the cleaning function
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"#\w+", "", text)             # remove hashtags
    text = re.sub(r"[^\w\s]", "", text)          # remove punctuation
    return text.strip()

# Apply to your DataFrame
df_full['Cleaned_Tweet'] = df_full['Tweet_text'].astype(str).apply(clean_tweet)

# Verify it worked
print(df_full[['Tweet_text','Cleaned_Tweet']].head())


                                          Tweet_text  \
0  Imperial Theatrical Coat for Court Lady https:...   
1  Half-length Figure of St Paul in an Oval. http...   
2  Great Exhibition Jurors&amp;#39; Medal https:/...   
3  Pair of candelabra https://t.co/KYopSWDSw2 htt...   
4  Banner (Nobori)\n http://t.co/yz34Xgo9a5 http:...   

                             Cleaned_Tweet  
0  imperial theatrical coat for court lady  
1  halflength figure of st paul in an oval  
2         great exhibition jurorsamp medal  
3                       pair of candelabra  
4                           banner noborin  


In [ ]:
from transformers import DistilBertTokenizer, DistilBertModel
import torch
import numpy as np
from tqdm.auto import tqdm

# 1. Load tokenizer & model on GPU
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model_distil = DistilBertModel.from_pretrained('distilbert-base-uncased') \
                              .eval().to('cuda')

# 2. Prepare text input
texts = df_full['Cleaned_Tweet'].astype(str).tolist()

# 3. Set a safe batch size for Google Colab
batch_size = 16
embeddings = []

# 4. Encode in batches and collect [CLS] embeddings
n_batches = (len(texts) + batch_size - 1) // batch_size
for i in tqdm(range(0, len(texts), batch_size), total=n_batches):
    batch = texts[i : i + batch_size]

    encodings = tokenizer(
        batch,
        padding='max_length',
        truncation=True,
        max_length=128,
        return_tensors='pt'
    )

    input_ids = encodings['input_ids'].to('cuda')
    attention_mask = encodings['attention_mask'].to('cuda')

    with torch.no_grad():
        outputs = model_distil(input_ids, attention_mask=attention_mask)

    # Extract CLS token embedding
    cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
    embeddings.append(cls_embeddings)

    # Optional: clear GPU cache
    torch.cuda.empty_cache()

# 5. Combine all CLS embeddings into one array
bert_features_full = np.vstack(embeddings)

# 6. Print the shape
print("✅ DistilBERT features shape:", bert_features_full.shape)

# 7. Optional: save features to disk
np.save("bert_features_full.npy", bert_features_full)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

  0%|          | 0/17481 [00:00<?, ?it/s]

✅ DistilBERT features shape: (279691, 768)


In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler

# 1. Select relevant behavioral features
behavior_cols = [
    'Word_Count', 'Url_Count', 'Retweet', 'Mentions_Count', 'Hashtags_Count',
    'Followers', 'Following'
]

# 2. Handle any missing values (if needed)
df_full[behavior_cols] = df_full[behavior_cols].fillna(0)

# 3. Extract and scale behavioral features
X_behav = df_full[behavior_cols].astype(float).values
scaler = StandardScaler()
X_behav_scaled = scaler.fit_transform(X_behav)

print("✅ Behavioral features shape:", X_behav_scaled.shape)


✅ Behavioral features shape: (279691, 7)


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split

# 1. Convert to torch tensors
X_text_tensor  = torch.tensor(bert_features_full, dtype=torch.float32)
X_behav_tensor = torch.tensor(X_behav_scaled, dtype=torch.float32)
y_tensor       = torch.tensor(df_full['Label'].values, dtype=torch.float32)

# 2. Custom Dataset
class FullHybridDataset(Dataset):
    def __init__(self, x_text, x_behav, y):
        self.x_text = x_text
        self.x_behav = x_behav
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.x_text[idx], self.x_behav[idx], self.y[idx]

dataset = FullHybridDataset(X_text_tensor, X_behav_tensor, y_tensor)

# 3. Split
train_size = int(0.8 * len(dataset))
val_size   = len(dataset) - train_size
train_ds, val_ds = random_split(dataset, [train_size, val_size])

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128)


In [ ]:
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.text_branch = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.behav_branch = nn.Sequential(
            nn.Linear(7, 16),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.classifier = nn.Sequential(
            nn.Linear(256 + 16, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x_text, x_behav):
        t = self.text_branch(x_text)
        b = self.behav_branch(x_behav)
        combined = torch.cat((t, b), dim=1)
        return self.classifier(combined)


In [ ]:
model = HybridModel().to('cuda')
loss_fn = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(5):
    model.train()
    total_loss = 0
    for xt, xb, yb in train_loader:
        xt, xb, yb = xt.to('cuda'), xb.to('cuda'), yb.to('cuda').unsqueeze(1)
        optimizer.zero_grad()
        preds = model(xt, xb)
        loss = loss_fn(preds, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1}/5 - Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/5 - Loss: 0.4640
Epoch 2/5 - Loss: 0.3495
Epoch 3/5 - Loss: 0.3163
Epoch 4/5 - Loss: 0.2961
Epoch 5/5 - Loss: 0.2815


In [ ]:
from sklearn.metrics import classification_report

model.eval()
all_preds, all_labels = [], []

with torch.no_grad():
    for xt, xb, yb in val_loader:
        xt, xb = xt.to('cuda'), xb.to('cuda')
        preds = model(xt, xb)
        preds = (preds.cpu().numpy() > 0.5).astype(int)
        all_preds.extend(preds.flatten())
        all_labels.extend(yb.numpy().flatten())

print(" Validation Accuracy Report:\n")
print(classification_report(all_labels, all_preds, digits=4))


 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.8906    0.8886    0.8896     27080
         1.0     0.8957    0.8976    0.8966     28859

    accuracy                         0.8932     55939
   macro avg     0.8932    0.8931    0.8931     55939
weighted avg     0.8932    0.8932    0.8932     55939



In [ ]:
from transformers import DistilBertModel

class HybridModelFineTune(torch.nn.Module):
    def __init__(self, dropout=0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.text_branch = torch.nn.Sequential(
            torch.nn.Linear(768, 256),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout)
        )
        self.behavior_branch = torch.nn.Sequential(
            torch.nn.Linear(7, 32),
            torch.nn.ReLU(),
            torch.nn.Dropout(dropout)
        )
        self.classifier = torch.nn.Sequential(
            torch.nn.Linear(256 + 32, 64),
            torch.nn.ReLU(),
            torch.nn.Linear(64, 1)
        )

    def forward(self, x_text, x_behav, attention_mask=None):
        cls_output = self.bert(x_text, attention_mask=attention_mask).last_hidden_state[:, 0]
        t = self.text_branch(cls_output)
        b = self.behavior_branch(x_behav)
        combined = torch.cat((t, b), dim=1)
        return self.classifier(combined)


In [ ]:
# ------------------- STEP 1: Upload & Load Dataset -------------------
import pandas as pd
from google.colab import files

uploaded = files.upload()  # Upload cleaned_twitter_data.csv

# Replace with actual uploaded filename if different
df_full = pd.read_csv("cleaned_twitter_data.csv")
df_full.head()


Saving cleaned_twitter_data.csv to cleaned_twitter_data.csv


,Twitter_User_Name,Twitter_Account,Twitter_User_Description,Tweet_id,Tweet_created_at,Tweet_text,Label,Word_Count,Url_Count,Retweet,...,Verb_Count,Adverb_Count,Positive_Word_Ratio,Negative_Word_Ratio,Neutral_Word_Ratio,Following,Followers,Verified,Link,Real_Location
0,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,8.027580e+17,27-11-2016 06:15,Imperial Theatrical Coat for Court Lady https:...,0,8,2,0,...,0,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
1,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,8.746920e+17,13-06-2017 18:15,Half-length Figure of St Paul in an Oval. http...,0,10,2,0,...,0,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
2,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,6.983900e+17,13-02-2016 06:15,Great Exhibition Jurors&amp;#39; Medal https:/...,0,6,2,0,...,0,0,0.125,0.0,0.875,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
3,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,6.976650e+17,11-02-2016 06:15,Pair of candelabra https://t.co/KYopSWDSw2 htt...,0,5,2,0,...,0,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
4,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,6.217450e+17,16-07-2015 18:15,Banner (Nobori)\n http://t.co/yz34Xgo9a5 http:...,0,4,2,0,...,1,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0


In [ ]:
import os
from pathlib import Path
import pandas as pd

# 1. List files in the root /content folder
print("Files in /content:")
print(os.listdir('/content'))

# 2. Search recursively for any CSV under /content
csv_files = list(Path('/content').rglob('*.csv'))
print("\nCSV files found:")
for path in csv_files:
    print(" ", path)

# 3. Load the dataset (adjust the path to whichever CSV you see)
dataset_path = '/content/cleaned_twitter_data.csv'
df_full = pd.read_csv(dataset_path)

# 4. Verify load
print("\nTotal tweets:", len(df_full))
print(df_full.columns)
print(df_full.head())


Files in /content:
['.config', 'cleaned_twitter_data.csv', 'sample_data']

CSV files found:
  /content/cleaned_twitter_data.csv
  /content/sample_data/california_housing_test.csv
  /content/sample_data/california_housing_train.csv
  /content/sample_data/mnist_test.csv
  /content/sample_data/mnist_train_small.csv

Total tweets: 279691
Index(['Twitter_User_Name', 'Twitter_Account', 'Twitter_User_Description',
       'Tweet_id', 'Tweet_created_at', 'Tweet_text', 'Label', 'Word_Count',
       'Url_Count', 'Retweet', 'Mentions_Count', 'Hashtags_Count',
       'QuesMark_Count', 'Exclamations_Count', 'SpecialCharacters_Count',
       'Nouns_Count', 'Pronouns_Count', 'Verb_Count', 'Adverb_Count',
       'Positive_Word_Ratio', 'Negative_Word_Ratio', 'Neutral_Word_Ratio',
       'Following', 'Followers', 'Verified', 'Link', 'Real_Location'],
      dtype='object')
  Twitter_User_Name Twitter_Account  \
0        Museum Bot       MuseumBot   
1        Museum Bot       MuseumBot   
2        Museum B

In [ ]:
import re

# Define the cleaning function
def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # remove URLs
    text = re.sub(r"@\w+", "", text)             # remove mentions
    text = re.sub(r"#\w+", "", text)             # remove hashtags
    text = re.sub(r"[^\w\s]", "", text)          # remove punctuation
    return text.strip()

# Apply to your DataFrame
df_full['Cleaned_Tweet'] = df_full['Tweet_text'].astype(str).apply(clean_tweet)

# Verify it worked
print(df_full[['Tweet_text','Cleaned_Tweet']].head())


                                          Tweet_text  \
0  Imperial Theatrical Coat for Court Lady https:...   
1  Half-length Figure of St Paul in an Oval. http...   
2  Great Exhibition Jurors&amp;#39; Medal https:/...   
3  Pair of candelabra https://t.co/KYopSWDSw2 htt...   
4  Banner (Nobori)\n http://t.co/yz34Xgo9a5 http:...   

                             Cleaned_Tweet  
0  imperial theatrical coat for court lady  
1  halflength figure of st paul in an oval  
2         great exhibition jurorsamp medal  
3                       pair of candelabra  
4                           banner noborin  


In [ ]:
# ------------------- STEP 2: Tokenize Tweets using DistilBERT -------------------
from transformers import DistilBertTokenizer

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

texts = df_full['Cleaned_Tweet'].astype(str).tolist()

# Tokenize all tweets in one go
encodings = tokenizer(
    texts,
    padding='max_length',
    truncation=True,
    max_length=128,
    return_tensors='pt'
)

input_ids      = encodings['input_ids']
attention_mask = encodings['attention_mask']


In [ ]:
import pandas as pd
import torch
import torch.nn as nn
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from transformers import DistilBertTokenizer, DistilBertModel
from torch.optim import AdamW
from torch.utils.data import TensorDataset, DataLoader


# Normalize behavioral features (optional but recommended)
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
behavior_features = scaler.fit_transform(behavior_features)

# === 2. Tokenize Text ===
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')

input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']
behavior_tensor = torch.tensor(behavior_features).float()
labels_tensor = torch.tensor(labels).float()

# === 3. Train/Test Split ===
train_idx, val_idx = train_test_split(range(len(labels)), test_size=0.2, random_state=42)

train_dataset = TensorDataset(
    input_ids[train_idx], attention_mask[train_idx],
    behavior_tensor[train_idx], labels_tensor[train_idx]
)
val_dataset = TensorDataset(
    input_ids[val_idx], attention_mask[val_idx],
    behavior_tensor[val_idx], labels_tensor[val_idx]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

# === 4. Hybrid Model Definition ===
class HybridModel(nn.Module):
    def __init__(self, behavior_dim):
        super(HybridModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.behavior_branch = nn.Sequential(
            nn.Linear(behavior_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16)
        )
        self.classifier = nn.Linear(768 + 16, 1)  # 768 from DistilBERT + 16 from behavior

    def forward(self, input_ids, attention_mask, behavior_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(bert_output.last_hidden_state[:, 0, :])  # CLS token
        behavior_output = self.behavior_branch(behavior_features)
        combined = torch.cat((cls_output, behavior_output), dim=1)
        logits = self.classifier(combined)
        return logits

# === 5. Training Setup ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel(behavior_dim=behavior_tensor.shape[1]).to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

best_val_acc = 0.0

# === 6. Training Loop ===
for epoch in range(7):  # ← you can increase/decrease
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
        optimizer.zero_grad()
        outputs = model(ids, masks, behaviors).squeeze()
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Training Loss: {avg_train_loss:.4f}")

    # === Validation ===
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
            outputs = model(ids, masks, behaviors).squeeze()
            preds.extend(torch.round(torch.sigmoid(outputs)).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    print("\n Validation Accuracy Report:\n")
    print(classification_report(true_labels, preds, digits=4))

    # Save best model
    from sklearn.metrics import accuracy_score
    val_acc = accuracy_score(true_labels, preds)
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_hybrid_model.pt")
        print("Best model saved!\n")

print(" Training complete.")


Epoch 1 - Training Loss: 0.2771

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9231    0.8890    0.9057     27010
         1.0     0.8998    0.9308    0.9150     28929

    accuracy                         0.9106     55939
   macro avg     0.9114    0.9099    0.9104     55939
weighted avg     0.9110    0.9106    0.9105     55939

Best model saved!

Epoch 2 - Training Loss: 0.1634

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9225    0.9167    0.9196     27010
         1.0     0.9226    0.9281    0.9254     28929

    accuracy                         0.9226     55939
   macro avg     0.9226    0.9224    0.9225     55939
weighted avg     0.9226    0.9226    0.9226     55939

Best model saved!

Epoch 3 - Training Loss: 0.1035

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9280    0.9180    0.9230     27010
         1

In [ ]:
from google.colab import files
# Click and select cleaned_twitter_data.csv (and best_hybrid_model.pt if you have it)
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


Saving cleaned_twitter_data.csv to cleaned_twitter_data.csv
Uploaded: ['cleaned_twitter_data.csv']


In [ ]:
import pandas as pd, os
csv_path = 'cleaned_twitter_data.csv'
assert os.path.exists(csv_path), f"{csv_path} not found — upload it first."

df = pd.read_csv(csv_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(3))


Rows: 279691
Columns: ['Twitter_User_Name', 'Twitter_Account', 'Twitter_User_Description', 'Tweet_id', 'Tweet_created_at', 'Tweet_text', 'Label', 'Word_Count', 'Url_Count', 'Retweet', 'Mentions_Count', 'Hashtags_Count', 'QuesMark_Count', 'Exclamations_Count', 'SpecialCharacters_Count', 'Nouns_Count', 'Pronouns_Count', 'Verb_Count', 'Adverb_Count', 'Positive_Word_Ratio', 'Negative_Word_Ratio', 'Neutral_Word_Ratio', 'Following', 'Followers', 'Verified', 'Link', 'Real_Location']


,Twitter_User_Name,Twitter_Account,Twitter_User_Description,Tweet_id,Tweet_created_at,Tweet_text,Label,Word_Count,Url_Count,Retweet,...,Verb_Count,Adverb_Count,Positive_Word_Ratio,Negative_Word_Ratio,Neutral_Word_Ratio,Following,Followers,Verified,Link,Real_Location
0,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,8.027580e+17,27-11-2016 06:15,Imperial Theatrical Coat for Court Lady https:...,0,8,2,0,...,0,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
1,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,8.746920e+17,13-06-2017 18:15,Half-length Figure of St Paul in an Oval. http...,0,10,2,0,...,0,0,0.000,0.0,1.000,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0
2,Museum Bot,MuseumBot,I am a bot that tweets a random high-res Open ...,6.983900e+17,13-02-2016 06:15,Great Exhibition Jurors&amp;#39; Medal https:/...,0,6,2,0,...,0,0,0.125,0.0,0.875,0,7816,0,https://twitter.com/MuseumBot?s=20,-1.0


In [ ]:
# === Prepare variables used in training code ===
TEXT_COL  = 'Tweet_text'
LABEL_COL = 'Label'
beh_cols  = ['Word_Count','Url_Count','Retweet','Mentions_Count','Hashtags_Count','Followers','Following']

# check columns
missing = [c for c in [TEXT_COL, LABEL_COL] + beh_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in CSV: {missing}. Edit TEXT_COL / beh_cols to match df.columns() output from previous cell.")

# Build arrays / lists
texts = df[TEXT_COL].astype(str).tolist()
# Convert label values to 0/1 integers (handles strings like 'phish' or '1')
def map_label(x):
    xs = str(x).strip().lower()
    if xs in ['1','true','yes','phish','phishing','malicious']: return 1
    if xs in ['0','false','no','benign','legit','legitimate']: return 0
    try:
        return int(float(xs))
    except:
        return 1 if xs not in ['', 'nan', 'none'] else 0

labels = [map_label(v) for v in df[LABEL_COL].tolist()]

import numpy as np
behavior_features = df[beh_cols].fillna(0).values.astype(float)  # shape (N,7)
print("Prepared: texts len", len(texts), "labels len", len(labels), "behavior_features shape", behavior_features.shape)


Prepared: texts len 279691 labels len 279691 behavior_features shape (279691, 7)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()
behavior_features = scaler.fit_transform(behavior_features)
# Save a copy of scaler for inference later
joblib.dump(scaler, 'scaler_demo.save')
print("Scaled behavior_features and saved scaler_demo.save")


Scaled behavior_features and saved scaler_demo.save


In [ ]:
from transformers import DistilBertTokenizer
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')

input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']
behavior_tensor = torch.tensor(behavior_features).float()
labels_tensor = torch.tensor(labels).float()

print("Tokenization complete. input_ids shape:", input_ids.shape, "behavior_tensor shape:", behavior_tensor.shape, "labels_tensor shape:", labels_tensor.shape)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

Tokenization complete. input_ids shape: torch.Size([279691, 128]) behavior_tensor shape: torch.Size([279691, 7]) labels_tensor shape: torch.Size([279691])


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

train_idx, val_idx = train_test_split(range(len(labels)), test_size=0.2, random_state=42)

train_dataset = TensorDataset(
    input_ids[train_idx], attention_mask[train_idx],
    behavior_tensor[train_idx], labels_tensor[train_idx]
)
val_dataset = TensorDataset(
    input_ids[val_idx], attention_mask[val_idx],
    behavior_tensor[val_idx], labels_tensor[val_idx]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
print("Loaders ready. Train batches:", len(train_loader), "Val batches:", len(val_loader))


Loaders ready. Train batches: 13985 Val batches: 3497


In [ ]:
# ------- Fix cell: re-import, recover behavior tensor, define + instantiate model -------

# 0. (Optional) install transformers if not present — uncomment if import fails
# !pip install -q transformers

# 1. imports
import os
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

# 2. Ensure tokenizer available (will download if needed)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Recover behavior_tensor or rebuild it from known variables if kernel was reset
# The training flow used variables named 'behavior_tensor' and 'behavior_features' and a dataframe 'df' with beh_cols.
# Attempt to use an existing behavior_tensor; otherwise build from behavior_features or df.

behavior_tensor_exists = 'behavior_tensor' in globals()
behavior_features_exists = 'behavior_features' in globals()
df_exists = 'df' in globals()
beh_cols_exists = 'beh_cols' in globals()

print("behavior_tensor in globals():", behavior_tensor_exists)
print("behavior_features in globals():", behavior_features_exists)
print("df in globals():", df_exists)
print("beh_cols in globals():", beh_cols_exists)

if not behavior_tensor_exists:
    if behavior_features_exists:
        # convert numpy array -> torch tensor
        try:
            behavior_tensor = torch.tensor(behavior_features, dtype=torch.float32)
            print("Rebuilt behavior_tensor from behavior_features with shape", behavior_tensor.shape)
        except Exception as e:
            print("Could not build behavior_tensor from behavior_features:", e)
    elif df_exists and beh_cols_exists:
        # build from df and beh_cols (most robust fallback)
        import numpy as np
        try:
            arr = df[beh_cols].fillna(0).values.astype(float)
            behavior_tensor = torch.tensor(arr, dtype=torch.float32)
            print("Built behavior_tensor from df[beh_cols] with shape", behavior_tensor.shape)
        except Exception as e:
            print("Failed to build behavior_tensor from df and beh_cols:", e)
    else:
        raise RuntimeError(
            "No behavior tensor/array found. Re-run the preprocessing cell that creates 'behavior_features' or ensure 'df' and 'beh_cols' are defined."
        )
else:
    print("Using existing behavior_tensor with shape", behavior_tensor.shape)

# 4. Determine behavior_dim for model instantiation
behavior_dim = behavior_tensor.shape[1] if hasattr(behavior_tensor, 'shape') else None
if behavior_dim is None:
    raise RuntimeError("Could not determine behavior_dim. Make sure behavior_tensor has shape (N, D).")
print("behavior_dim =", behavior_dim)

# 5. Define HybridModel (same architecture used before)
class HybridModel(nn.Module):
    def __init__(self, behavior_dim):
        super(HybridModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.behavior_branch = nn.Sequential(
            nn.Linear(behavior_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16)
        )
        self.classifier = nn.Linear(768 + 16, 1)

    def forward(self, input_ids, attention_mask, behavior_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(bert_output.last_hidden_state[:, 0, :])
        behavior_output = self.behavior_branch(behavior_features)
        combined = torch.cat((cls_output, behavior_output), dim=1)
        logits = self.classifier(combined)
        return logits

# 6. Instantiate model on device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel(behavior_dim=behavior_dim).to(device)
print("Model instantiated on", device)

# 7. If you have a saved checkpoint in working dir, load it now
ckpt = "best_hybrid_model.pt"
if os.path.exists(ckpt):
    try:
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state)
        print("Loaded state_dict from", ckpt)
    except Exception as e:
        print("Could not load state_dict directly:", e)
        try:
            # try loading as a full model object
            model = torch.load(ckpt, map_location=device)
            print("Loaded full model object from", ckpt)
        except Exception as e2:
            print("Second load attempt failed:", e2)
else:
    print(f"No checkpoint found at '{ckpt}' in working dir. If you want predictions, upload the file now.")

# 8. Set model to eval mode
model.eval()


torch: 2.8.0+cu126 cuda: True
behavior_tensor in globals(): True
behavior_features in globals(): True
df in globals(): True
beh_cols in globals(): True
Using existing behavior_tensor with shape torch.Size([279691, 7])
behavior_dim = 7


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Model instantiated on cuda
No checkpoint found at 'best_hybrid_model.pt' in working dir. If you want predictions, upload the file now.


HybridModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_

In [ ]:
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self, behavior_dim):
        super(HybridModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.behavior_branch = nn.Sequential(
            nn.Linear(behavior_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16)
        )
        self.classifier = nn.Linear(768 + 16, 1)

    def forward(self, input_ids, attention_mask, behavior_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(bert_output.last_hidden_state[:, 0, :])
        behavior_output = self.behavior_branch(behavior_features)
        combined = torch.cat((cls_output, behavior_output), dim=1)
        logits = self.classifier(combined)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel(behavior_dim=behavior_tensor.shape[1]).to(device)
print("Model instantiated on device:", device)


Model instantiated on device: cuda


In [ ]:
from torch.optim import AdamW
import torch

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

best_val_acc = 0.0

for epoch in range(2):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
        optimizer.zero_grad()
        outputs = model(ids, masks, behaviors).squeeze()
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Training Loss: {avg_train_loss:.4f}")

    # Validation
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
            outputs = model(ids, masks, behaviors).squeeze()
            preds.extend(torch.round(torch.sigmoid(outputs)).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    from sklearn.metrics import classification_report, accuracy_score
    val_acc = accuracy_score(true_labels, preds)
    print("\n Validation Accuracy Report:\n")
    print(classification_report(true_labels, preds, digits=4))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_hybrid_model.pt")
        print("Best model saved!\n")

print("Training complete. Best val acc:", best_val_acc)


Epoch 1 - Training Loss: 0.1580

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9559    0.9599    0.9579     27010
         1.0     0.9624    0.9587    0.9605     28929

    accuracy                         0.9593     55939
   macro avg     0.9592    0.9593    0.9592     55939
weighted avg     0.9593    0.9593    0.9593     55939

Best model saved!

Epoch 2 - Training Loss: 0.0784

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9687    0.9596    0.9641     27010
         1.0     0.9626    0.9710    0.9668     28929

    accuracy                         0.9655     55939
   macro avg     0.9657    0.9653    0.9655     55939
weighted avg     0.9656    0.9655    0.9655     55939

Best model saved!

Training complete. Best val acc: 0.9655338851248682


In [ ]:
import torch
import numpy as np
from tqdm.notebook import tqdm

def predict_on_texts(texts, beh_vectors, batch_size=16):
    """
    texts: list[str]
    beh_vectors: list[list[float]] (same order)
    returns: list of probabilities (floats)
    """
    probs = []
    model.to(device)
    model.eval()
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_beh = beh_vectors[i:i+batch_size]
        enc = tokenizer(batch_texts, padding='max_length', truncation=True, max_length=128, return_tensors='pt')
        input_ids = enc['input_ids'].to(device)
        attention_mask = enc['attention_mask'].to(device)
        beh_scaled = scaler.transform(batch_beh)
        beh_tensor = torch.tensor(beh_scaled, dtype=torch.float32).to(device)
        with torch.no_grad():
            logits = model(input_ids, attention_mask, beh_tensor).squeeze()
            batch_probs = torch.sigmoid(logits).cpu().numpy()
        # normalize output shape
        if np.isscalar(batch_probs):
            probs.append(float(batch_probs))
        else:
            probs.extend(batch_probs.tolist())
    return probs


In [ ]:
N = 5
sample = df.sample(n=N, random_state=42).reset_index(drop=False)  # keep original index in column 'index'
texts = sample['Tweet_text'].astype(str).tolist()
beh_cols = ['Word_Count','Url_Count','Retweet','Mentions_Count','Hashtags_Count','Followers','Following']
beh = sample[beh_cols].fillna(0).values.tolist()

probs = predict_on_texts(texts, beh, batch_size=8)
sample['pred_prob'] = probs
sample['pred_label'] = (sample['pred_prob'] > 0.5).astype(int)
# Display relevant columns neatly
display(sample[['index','Tweet_text','pred_prob','pred_label','Label']])


,index,Tweet_text,pred_prob,pred_label,Label
0,27874,According to @MassStatePolice they are hoping ...,0.997891,1,1
1,108353,#IslamicTeachings\nWe r a religion of peace ht...,0.999418,1,1
2,57687,@ReptarAzar Eh. Just trying to give you a hard...,0.999293,1,1
3,86981,VHDL or Verilog?... https://t.co/3Tf3LNNltg,0.228536,0,0
4,147869,"Winklevoss stays bullish on bitcoin, hires NYS...",0.000335,0,0


In [ ]:
per_class = 3
phish = df[df['Label'].astype(int)==1].sample(n=per_class, random_state=1)
benign = df[df['Label'].astype(int)==0].sample(n=per_class, random_state=1)
balanced = pd.concat([phish, benign]).reset_index(drop=False)
texts = balanced['Tweet_text'].astype(str).tolist()
beh = balanced[beh_cols].fillna(0).values.tolist()

probs = predict_on_texts(texts, beh, batch_size=8)
balanced['pred_prob'] = probs
balanced['pred_label'] = (balanced['pred_prob'] > 0.5).astype(int)
display(balanced[['index','Tweet_text','pred_prob','pred_label','Label']])


,index,Tweet_text,pred_prob,pred_label,Label
0,49121,RT @kourtneykardash: Happy Sunday! https://t.c...,0.999599,1,1
1,150705,On Iqbal Day let us remember his message: You ...,0.999846,1,1
2,96862,.@EatFellowHumans still can't remember why we ...,0.997211,1,1
3,269124,LMAO \xf0\x9f\x98\x86 This made me spit out my...,0.001427,0,0
4,250512,"My Favorite Things, Round 1010",0.000340,0,0
5,75756,@inky A woodchuck would chuck all the wood he ...,0.026024,0,0


In [ ]:
# Load scaler (if you restarted session earlier, re-load tokenizer/model/scaler first)
import joblib, time
scaler = joblib.load('scaler_demo.save')

def predict_one(tweet_text, beh_vector, threshold=0.5):
    enc = tokenizer([tweet_text], padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    beh_scaled = scaler.transform([beh_vector])
    beh_tensor = torch.tensor(beh_scaled, dtype=torch.float32).to(device)
    with torch.no_grad():
        logits = model(input_ids, attention_mask, beh_tensor).squeeze()
        prob = torch.sigmoid(logits).item()
    return prob

# Examples (adjust behavioral vectors to realistic values)
examples = [
    ("Verify your account immediately at http://bit.ly/fake", [10,1,0,0,0,1,2]),
    ("Had a great day today with friends #weekend", [8,0,1,0,1,520,400])
]
for text, beh in examples:
    t0 = time.time()
    p = predict_one(text, beh)
    t1 = time.time()
    print(f"Text: {text}\nProb(phish)={p:.4f}, Pred={(1 if p>0.5 else 0)}, latency={(t1-t0)*1000:.1f} ms\n")


Text: Verify your account immediately at http://bit.ly/fake
Prob(phish)=0.0396, Pred=0, latency=39.9 ms

Text: Had a great day today with friends #weekend
Prob(phish)=0.0790, Pred=0, latency=9.5 ms



In [ ]:
# For speed, evaluate on a sample or your validation split
eval_df = df.sample(n=2000, random_state=2)   # or use your val split if available
texts = eval_df['Tweet_text'].astype(str).tolist()
beh = eval_df[beh_cols].fillna(0).values.tolist()

probs = predict_on_texts(texts, beh, batch_size=16)
eval_df = eval_df.reset_index(drop=False)
eval_df['pred_prob'] = probs
topk = eval_df.sort_values('pred_prob', ascending=False).head(10)
display(topk[['index','Tweet_text','pred_prob','Label']])


,index,Tweet_text,pred_prob,Label
1299,48957,RT @perfectkhloek: So thankful for @khloekarda...,0.999881,1
1310,49768,Who's ready for a brand new episode of #Reveng...,0.999881,1
1654,97113,RT @FoodNetwork: This Smoked Rib Platter is ju...,0.999880,1
1337,174565,RT @Anka26102000: .@1LIVE I want to hear @Mike...,0.999878,1
605,102298,RT @danieldraper: You can\xe2\x80\x99t take ou...,0.999878,1
546,48270,RT @khloekardashian: Tonight on #KUWTK 9/8c on...,0.999878,1
1113,49151,RT @kimkslays: I love the fact @khloekardashia...,0.999878,1
1037,12554,RT @MuslimCouncil: Great to see @SadiqKhan sho...,0.999877,1
1907,99302,in case ya missed fridays new #DDD - its back ...,0.999876,1
1668,12572,Out with the fantastic campaign team in Harrow...,0.999874,1


In [ ]:
from google.colab import files
# Click and select cleaned_twitter_data.csv (and best_hybrid_model.pt if you have it)
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving merged_twitter_phishing_dataset.csv to merged_twitter_phishing_dataset.csv
Uploaded: ['merged_twitter_phishing_dataset.csv']


In [ ]:
from google.colab import files
# Click and select cleaned_twitter_data.csv (and best_hybrid_model.pt if you have it)
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))

Saving merged_twitter_with_synthetic.csv to merged_twitter_with_synthetic.csv
Uploaded: ['merged_twitter_with_synthetic.csv']


In [ ]:
import pandas as pd
import os

csv_path = 'merged_twitter_with_synthetic.csv'   # Updated file name
assert os.path.exists(csv_path), f"{csv_path} not found — upload it first."

df = pd.read_csv(csv_path)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head(3))


/tmp/ipython-input-654479599.py:7: DtypeWarning: Columns (3,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(csv_path)


Rows: 279691
Columns: ['Twitter_User_Name', 'Twitter_Account', 'Twitter_User_Description', 'Tweet_id', 'Tweet_created_at', 'Tweet_text', 'Label', 'Word_Count', 'Url_Count', 'Retweet', 'Mentions_Count', 'Hashtags_Count', 'QuesMark_Count', 'Exclamations_Count', 'SpecialCharacters_Count', 'Nouns_Count', 'Pronouns_Count', 'Verb_Count', 'Adverb_Count', 'Positive_Word_Ratio', 'Negative_Word_Ratio', 'Neutral_Word_Ratio', 'Following', 'Followers', 'Verified', 'Link', 'Real_Location']


,Twitter_User_Name,Twitter_Account,Twitter_User_Description,Tweet_id,Tweet_created_at,Tweet_text,Label,Word_Count,Url_Count,Retweet,...,Verb_Count,Adverb_Count,Positive_Word_Ratio,Negative_Word_Ratio,Neutral_Word_Ratio,Following,Followers,Verified,Link,Real_Location
0,Sadiq Khan,SadiqKhan,Mayor of London. #TeamKhan,978976000000000000.0,28-03-2018 12:43,Today we unveiled the latest artwork to be fea...,1,21,1,0,...,3,0,0.000000,0.0,1.000000,10500,1100000,1,https://twitter.com/SadiqKhan?s=20,1.0
1,Sadiq Khan,SadiqKhan,Mayor of London. #TeamKhan,1010100000000000000.0,22-06-2018 09:50,The Empire Windrush arrived on our shores 70 y...,1,21,1,0,...,3,1,0.000000,0.0,1.000000,10500,1100000,1,https://twitter.com/SadiqKhan?s=20,1.0
2,Sadiq Khan,SadiqKhan,Mayor of London. #TeamKhan,1004650000000000000.0,07-06-2018 09:05,RT @coyleneil: Great piece by London Mayor @Sa...,1,22,1,1,...,0,0,0.071429,0.0,0.928571,10500,1100000,1,https://twitter.com/SadiqKhan?s=20,1.0


In [ ]:
# === Prepare variables used in training code ===
TEXT_COL  = 'Tweet_text'
LABEL_COL = 'Label'
beh_cols  = ['Word_Count','Url_Count','Retweet','Mentions_Count','Hashtags_Count','Followers','Following']

# check columns
missing = [c for c in [TEXT_COL, LABEL_COL] + beh_cols if c not in df.columns]
if missing:
    raise KeyError(f"Missing columns in CSV: {missing}. Edit TEXT_COL / beh_cols to match df.columns() output from previous cell.")

# Build arrays / lists
texts = df[TEXT_COL].astype(str).tolist()
# Convert label values to 0/1 integers (handles strings like 'phish' or '1')
def map_label(x):
    xs = str(x).strip().lower()
    if xs in ['1','true','yes','phish','phishing','malicious']: return 1
    if xs in ['0','false','no','benign','legit','legitimate']: return 0
    try:
        return int(float(xs))
    except:
        return 1 if xs not in ['', 'nan', 'none'] else 0

labels = [map_label(v) for v in df[LABEL_COL].tolist()]

import numpy as np
behavior_features = df[beh_cols].fillna(0).values.astype(float)  # shape (N,7)
print("Prepared: texts len", len(texts), "labels len", len(labels), "behavior_features shape", behavior_features.shape)


Prepared: texts len 279691 labels len 279691 behavior_features shape (279691, 7)


In [ ]:
from sklearn.preprocessing import MinMaxScaler
import joblib

scaler = MinMaxScaler()
behavior_features = scaler.fit_transform(behavior_features)
# Save a copy of scaler for inference later
joblib.dump(scaler, 'scaler_demo.save')
print("Scaled behavior_features and saved scaler_demo.save")


Scaled behavior_features and saved scaler_demo.save


In [ ]:
from transformers import DistilBertTokenizer
import torch

tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
encodings = tokenizer(texts, truncation=True, padding=True, max_length=128, return_tensors='pt')

input_ids = encodings['input_ids']
attention_mask = encodings['attention_mask']
behavior_tensor = torch.tensor(behavior_features).float()
labels_tensor = torch.tensor(labels).float()

print("Tokenization complete. input_ids shape:", input_ids.shape, "behavior_tensor shape:", behavior_tensor.shape, "labels_tensor shape:", labels_tensor.shape)


Tokenization complete. input_ids shape: torch.Size([279691, 128]) behavior_tensor shape: torch.Size([279691, 7]) labels_tensor shape: torch.Size([279691])


In [ ]:
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

train_idx, val_idx = train_test_split(range(len(labels)), test_size=0.2, random_state=42)

train_dataset = TensorDataset(
    input_ids[train_idx], attention_mask[train_idx],
    behavior_tensor[train_idx], labels_tensor[train_idx]
)
val_dataset = TensorDataset(
    input_ids[val_idx], attention_mask[val_idx],
    behavior_tensor[val_idx], labels_tensor[val_idx]
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)
print("Loaders ready. Train batches:", len(train_loader), "Val batches:", len(val_loader))


Loaders ready. Train batches: 13985 Val batches: 3497


In [ ]:
# ------- Fix cell: re-import, recover behavior tensor, define + instantiate model -------

# 0. (Optional) install transformers if not present — uncomment if import fails
# !pip install -q transformers

# 1. imports
import os
import torch
import torch.nn as nn
from transformers import DistilBertModel, DistilBertTokenizer

print("torch:", torch.__version__, "cuda:", torch.cuda.is_available())

# 2. Ensure tokenizer available (will download if needed)
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')

# 3. Recover behavior_tensor or rebuild it from known variables if kernel was reset
# The training flow used variables named 'behavior_tensor' and 'behavior_features' and a dataframe 'df' with beh_cols.
# Attempt to use an existing behavior_tensor; otherwise build from behavior_features or df.

behavior_tensor_exists = 'behavior_tensor' in globals()
behavior_features_exists = 'behavior_features' in globals()
df_exists = 'df' in globals()
beh_cols_exists = 'beh_cols' in globals()

print("behavior_tensor in globals():", behavior_tensor_exists)
print("behavior_features in globals():", behavior_features_exists)
print("df in globals():", df_exists)
print("beh_cols in globals():", beh_cols_exists)

if not behavior_tensor_exists:
    if behavior_features_exists:
        # convert numpy array -> torch tensor
        try:
            behavior_tensor = torch.tensor(behavior_features, dtype=torch.float32)
            print("Rebuilt behavior_tensor from behavior_features with shape", behavior_tensor.shape)
        except Exception as e:
            print("Could not build behavior_tensor from behavior_features:", e)
    elif df_exists and beh_cols_exists:
        # build from df and beh_cols (most robust fallback)
        import numpy as np
        try:
            arr = df[beh_cols].fillna(0).values.astype(float)
            behavior_tensor = torch.tensor(arr, dtype=torch.float32)
            print("Built behavior_tensor from df[beh_cols] with shape", behavior_tensor.shape)
        except Exception as e:
            print("Failed to build behavior_tensor from df and beh_cols:", e)
    else:
        raise RuntimeError(
            "No behavior tensor/array found. Re-run the preprocessing cell that creates 'behavior_features' or ensure 'df' and 'beh_cols' are defined."
        )
else:
    print("Using existing behavior_tensor with shape", behavior_tensor.shape)

# 4. Determine behavior_dim for model instantiation
behavior_dim = behavior_tensor.shape[1] if hasattr(behavior_tensor, 'shape') else None
if behavior_dim is None:
    raise RuntimeError("Could not determine behavior_dim. Make sure behavior_tensor has shape (N, D).")
print("behavior_dim =", behavior_dim)

# 5. Define HybridModel (same architecture used before)
class HybridModel(nn.Module):
    def __init__(self, behavior_dim):
        super(HybridModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.behavior_branch = nn.Sequential(
            nn.Linear(behavior_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16)
        )
        self.classifier = nn.Linear(768 + 16, 1)

    def forward(self, input_ids, attention_mask, behavior_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(bert_output.last_hidden_state[:, 0, :])
        behavior_output = self.behavior_branch(behavior_features)
        combined = torch.cat((cls_output, behavior_output), dim=1)
        logits = self.classifier(combined)
        return logits

# 6. Instantiate model on device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel(behavior_dim=behavior_dim).to(device)
print("Model instantiated on", device)

# 7. If you have a saved checkpoint in working dir, load it now
ckpt = "best_hybrid_model.pt"
if os.path.exists(ckpt):
    try:
        state = torch.load(ckpt, map_location=device)
        model.load_state_dict(state)
        print("Loaded state_dict from", ckpt)
    except Exception as e:
        print("Could not load state_dict directly:", e)
        try:
            # try loading as a full model object
            model = torch.load(ckpt, map_location=device)
            print("Loaded full model object from", ckpt)
        except Exception as e2:
            print("Second load attempt failed:", e2)
else:
    print(f"No checkpoint found at '{ckpt}' in working dir. If you want predictions, upload the file now.")

# 8. Set model to eval mode
model.eval()


torch: 2.8.0+cu126 cuda: True
behavior_tensor in globals(): True
behavior_features in globals(): True
df in globals(): True
beh_cols in globals(): True
Using existing behavior_tensor with shape torch.Size([279691, 7])
behavior_dim = 7
Model instantiated on cuda
Loaded state_dict from best_hybrid_model.pt


HybridModel(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSdpaAttention(
            (dropout): Dropout(p=0.1, inplace=False)
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in_

In [ ]:
import torch.nn as nn

class HybridModel(nn.Module):
    def __init__(self, behavior_dim):
        super(HybridModel, self).__init__()
        self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
        self.dropout = nn.Dropout(0.3)
        self.behavior_branch = nn.Sequential(
            nn.Linear(behavior_dim, 32),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(32, 16)
        )
        self.classifier = nn.Linear(768 + 16, 1)

    def forward(self, input_ids, attention_mask, behavior_features):
        bert_output = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_output = self.dropout(bert_output.last_hidden_state[:, 0, :])
        behavior_output = self.behavior_branch(behavior_features)
        combined = torch.cat((cls_output, behavior_output), dim=1)
        logits = self.classifier(combined)
        return logits

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = HybridModel(behavior_dim=behavior_tensor.shape[1]).to(device)
print("Model instantiated on device:", device)


Model instantiated on device: cuda


In [ ]:
from torch.optim import AdamW
import torch

criterion = nn.BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

best_val_acc = 0.0

for epoch in range(1):
    model.train()
    total_loss = 0

    for batch in train_loader:
        ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
        optimizer.zero_grad()
        outputs = model(ids, masks, behaviors).squeeze()
        loss = criterion(outputs, labels_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch+1} - Training Loss: {avg_train_loss:.4f}")

    # Validation
    model.eval()
    preds, true_labels = [], []

    with torch.no_grad():
        for batch in val_loader:
            ids, masks, behaviors, labels_batch = [x.to(device) for x in batch]
            outputs = model(ids, masks, behaviors).squeeze()
            preds.extend(torch.round(torch.sigmoid(outputs)).cpu().numpy())
            true_labels.extend(labels_batch.cpu().numpy())

    from sklearn.metrics import classification_report, accuracy_score
    val_acc = accuracy_score(true_labels, preds)
    print("\n Validation Accuracy Report:\n")
    print(classification_report(true_labels, preds, digits=4))

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best_hybrid_model.pt")
        print("Best model saved!\n")

print("Training complete. Best val acc:", best_val_acc)


Epoch 1 - Training Loss: 0.1585

 Validation Accuracy Report:

              precision    recall  f1-score   support

         0.0     0.9686    0.9392    0.9537     27166
         1.0     0.9442    0.9712    0.9575     28773

    accuracy                         0.9557     55939
   macro avg     0.9564    0.9552    0.9556     55939
weighted avg     0.9560    0.9557    0.9556     55939

Best model saved!

Training complete. Best val acc: 0.9556659933141458


In [ ]:
import time
import torch

# Function to predict one tweet
def predict_one(tweet_text, beh_vector, threshold=0.6):
    enc = tokenizer([tweet_text], padding='max_length', truncation=True, max_length=128, return_tensors='pt')
    input_ids = enc['input_ids'].to(device)
    attention_mask = enc['attention_mask'].to(device)
    beh_scaled = scaler.transform([beh_vector])
    beh_tensor = torch.tensor(beh_scaled, dtype=torch.float32).to(device)

    with torch.no_grad():
        logits = model(input_ids, attention_mask, beh_tensor).squeeze()
        prob = torch.sigmoid(logits).item()
        pred = 1 if prob > threshold else 0

    return prob, pred

# Demo examples
examples = [
    ("Verify your account immediately at http://bit.ly/fake", [10,1,0,0,0,1,2]),   # phishing
    ("Had a great day today with friends #weekend", [8,0,1,0,1,520,400]),           # non-phishing
    ("You won a prize! Claim now: http://tinyurl.com/free-gift", [12,1,0,0,0,5,10]),# phishing
    ("Lunch at the new cafe was amazing!", [9,0,0,0,0,300,350]),                   # non-phishing
    ("Security alert: Reset password immediately: http://verify-twitter.net", [11,1,0,0,0,2,5]) # phishing
]

# Run demo
for text, beh in examples:
    t0 = time.time()
    prob, pred = predict_one(text, beh)
    t1 = time.time()
    print(f"Text: {text}")
    print(f"Prob(phish)={prob:.4f}, Pred={pred}, latency={(t1-t0)*1000:.1f} ms\n")


Text: Verify your account immediately at http://bit.ly/fake
Prob(phish)=0.9589, Pred=1, latency=12.2 ms

Text: Had a great day today with friends #weekend
Prob(phish)=0.5176, Pred=0, latency=9.3 ms

Text: You won a prize! Claim now: http://tinyurl.com/free-gift
Prob(phish)=0.9997, Pred=1, latency=9.0 ms

Text: Lunch at the new cafe was amazing!
Prob(phish)=0.2508, Pred=0, latency=8.8 ms

Text: Security alert: Reset password immediately: http://verify-twitter.net
Prob(phish)=0.9997, Pred=1, latency=8.8 ms

